# 01 — Ingest

Pulls Chicago crime records from the city's Socrata ([SODA](https://dev.socrata.com/)) API — `ijzp-q8t2`, the "Crimes 2001 to Present" dataset — filtered to `date >= 2019-01-01`, and caches the raw result to `data/crimes_raw.parquet` for the rest of the pipeline. Uses anonymous (no app token) access, so a fresh run is subject to Socrata's public rate limits and can take a while; see the retry/backoff logic below.

In [ ]:
import pandas as pd
import sodapy
print("Environment is ready")

## Connectivity check

Pull 5 rows with no filters, just to confirm the endpoint and column names before running the full paginated fetch below.

In [ ]:
from sodapy import Socrata

client = Socrata("data.cityofchicago.org", None, timeout=60)

# tiny test pull — just 5 rows
test = client.get("ijzp-q8t2", limit=5)
test_df = pd.DataFrame.from_records(test)
print(test_df.columns.tolist())
test_df.head()

## Full pull

Page through the entire `2019-01-01` onward range in 50k-row batches. Each page retries up to 5 times with linear backoff before the run gives up — the Socrata API is flaky under load, and a fresh pull can take a while.

In [ ]:
import pandas as pd
import time
from sodapy import Socrata

client = Socrata("data.cityofchicago.org", None, timeout=120)

all_rows = []
offset = 0
page_size = 50_000
max_retries = 5

while True:
    # try the same page up to max_retries times before giving up
    for attempt in range(max_retries):
        try:
            page = client.get(
                "ijzp-q8t2",
                where="date >= '2019-01-01T00:00:00'",
                limit=page_size,
                offset=offset,
                order="date"
            )
            break  # success — exit the retry loop
        except Exception as e:
            wait = 10 * (attempt + 1)   # back off: 10s, 20s, 30s...
            print(f"  page at offset {offset:,} failed ({type(e).__name__}), "
                  f"retry {attempt + 1}/{max_retries} in {wait}s")
            time.sleep(wait)
    else:
        # all retries exhausted
        raise RuntimeError(f"Failed at offset {offset:,} after {max_retries} retries")

    if not page:
        break
    all_rows.extend(page)
    offset += page_size
    print(f"Pulled {len(all_rows):,} rows so far...")

df = pd.DataFrame.from_records(all_rows)
print(f"\nDone. Total: {len(df):,} rows")

## Sanity checks

Confirm the pull covers the expected date range and that the crime-type and coordinate columns look usable before saving.

In [ ]:
df = pd.DataFrame.from_records(all_rows)
print(f"Recovered {len(df):,} rows")
print(df["date"].min(), "→", df["date"].max())

In [ ]:
print(df["primary_type"].nunique(), "crime types")
print(df[["latitude", "longitude"]].isna().sum())

## Save

Cache the pulled data to Parquet so later notebooks (and reruns of this one) don't have to re-hit the API.

In [ ]:
df.to_parquet("data/crimes_raw.parquet", index=False)
print("Saved to data/crimes_raw.parquet")

In [ ]:
import os
print(os.listdir("data"))
size_mb = os.path.getsize("data/crimes_raw.parquet") / 1e6
print(f"File size: {size_mb:.1f} MB")